# Mental Health — A Flat Table Holding Twelve Classifications

CDC WONDER refuses automated query. It returns HTTP 403 to this client, and the corpus's source
survey recorded the domain as blocked on that basis. Much of the same NCHS material is published
through `data.cdc.gov`'s Socrata API, which does not refuse — so this notebook is built there,
and it is careful to say that the API is **not a WONDER substitute**: WONDER's custom
cross-tabulations are not available through it.

What the API serves has a structure worth naming. The NCHS suicide death-rate table is
**6,390 rows** spanning **1950 to 2018**, and its `stub_name` column does not hold strata of one
breakdown. It holds **twelve different classification schemes of the same deaths** — Total, Sex,
Age, Sex and age, Sex and race, Sex and race and Hispanic origin, and so on — of which
**only 84 rows (1.3%) are the whole population.**

In 2018 alone, **276 rows** describe the same twelve months. Two of them are the population.

Three further things are in the table and not in its column names. A second column, `unit`,
**crosses all twelve schemes** with two incompatible rate definitions — age-adjusted and crude —
so filtering to `Total` still leaves two rows per year. Two of the twelve schemes use an
**incompatible race-coding standard**, and **2018 is the year both appear** — 162 rows on the
legacy standard beside 114 on the new one, describing the same deaths. And the year axis is not
annual: sixty-eight years of span carry only **42 year-points**, decadal until 1980 and annual
after.

A mean over the estimate column gives **13.71**. The 2018 age-adjusted rate is **14.20** and the
crude rate is **14.80**. The first number is not a rate for anything: it averages a national
population against sex-by-age-by-race-by-Hispanic-origin cells, and age-adjusted rates against
crude ones, weighting each equally.

**On the subject matter.** This notebook is about the structure of a published mortality table.
It makes no claim about causes, trends or policy, and none of what follows should be read as
analysis of suicide itself.

## Learning Objectives

1. **Check what a classification column actually holds** before grouping or averaging on it.
2. **Recognise a table that mixes aggregation levels**, and see why no single filter on it is
   optional.
3. **Check whether a second column crosses the first** — a correct filter on one partition can
   still leave two incompatible rows per key.
4. **Spot a coding-standard changeover** and know that overlapping years are not a longer series.
5. **Read the year axis before plotting it**, when a span and a point count disagree.
6. **Pass an explicit page limit to a paginated API**, and refuse a result that exactly fills it.

## Prerequisites

- Handbook §3 (denominators, universes and partitions).
- D20 (`B28001`), the nearest relative — two partitions in one flat list. This table has twelve.
- Handbook §1.7 (HUD CHAS) for the other case of a published table whose categories changed
  underneath a stable column name.

## Data Sources

**CDC / NCHS, *Health, United States* — death rates for suicide**, resource `9j2v-jamp` on
`data.cdc.gov`, retrieved through the Socrata API.

**Why not WONDER.** CDC WONDER is the canonical route to NCHS mortality data and it refuses
automated query — HTTP 403, re-checked on 2026-09-15. This is a property of the service, not a
failure here, and it is why the corpus's survey recorded D28 as blocked. The Socrata route
reaches published NCHS tables and not WONDER's query engine; anything requiring a custom
cross-tabulation still needs WONDER and a human.

**One transport note.** Socrata's default page is 1,000 rows. The client passes `$limit`
explicitly and **raises if the response exactly fills it**, because that is the single case where
a complete result and a truncated one are indistinguishable.

## 1. Setup

In [1]:
"""Configuration."""
from __future__ import annotations

import numpy as np
import pandas as pd

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.sources import (
    verify_digest,
    CDC_SOCRATA_MAX_LIMIT,
    NCHS_SINGLE_RACE_MARKER,
    NCHS_SUICIDE_DATASET,
    NCHS_TOTAL_STUB,
    describe_stub_levels,
    fetch_cdc_socrata,
)

#: The measure column, and the one an analyst would plot.
ESTIMATE = "estimate"

pd.set_option("display.width", 112)
print(f"dataset       : data.cdc.gov/{NCHS_SUICIDE_DATASET}")
print(f"total stub    : {NCHS_TOTAL_STUB!r}")
print(f"race marker   : {NCHS_SINGLE_RACE_MARKER!r}")
print(f"request limit : {CDC_SOCRATA_MAX_LIMIT:,} (Socrata's own default is 1,000)")

#: The digest of the payload every figure below was computed from, on this
#: record's own hash basis. The publisher serves no prior vintage, so this pin
#: does not restore one -- it guarantees an upstream change cannot pass
#: unnoticed: the notebook raises instead of reporting different numbers under
#: unchanged prose. Handbook §5.2.
PINNED_NCHS_SUICIDE = "45e36dd2525a2e52232057cb4563df48747e910d7541f533242d43b0132e2f22"


dataset       : data.cdc.gov/9j2v-jamp
total stub    : 'Total'
race marker   : '(Single race)'
request limit : 50,000 (Socrata's own default is 1,000)


## 2. Retrieval

In [2]:
"""Retrieve. WONDER refuses automated query; this API does not."""
deaths, prov = verify_digest(
    fetch_cdc_socrata(dataset=NCHS_SUICIDE_DATASET),
    PINNED_NCHS_SUICIDE, label='NCHS 9j2v-jamp')
deaths[ESTIMATE] = pd.to_numeric(deaths[ESTIMATE], errors="coerce")
deaths["year"] = pd.to_numeric(deaths["year"], errors="coerce")

years = sorted(int(y) for y in deaths["year"].unique())
gaps = [(a, b) for a, b in zip(years, years[1:]) if b - a > 1]

print(f"rows        : {len(deaths):,}")
print(f"columns     : {len(deaths.columns)}")
print(f"year range  : {years[0]} to {years[-1]}  (span {years[-1] - years[0]} years)")
print(f"year points : {len(years)}")
print(f"gaps        : {gaps}")
print(f"indicator   : {', '.join(deaths['indicator'].unique())}")
print(f"unit        : {deaths['unit'].nunique()} distinct rate definitions")
print()
for note in prov.notes[1:]:
    print(f"  - {note}")


rows        : 6,390
columns     : 13
year range  : 1950 to 2018  (span 68 years)
year points : 42
gaps        : [(1950, 1960), (1960, 1970), (1970, 1980)]
indicator   : Death rates for suicide
unit        : 2 distinct rate definitions

  - CDC WONDER refuses automated query; this API does not, and serves much of the same NCHS material. It is not a WONDER substitute -- WONDER's custom cross-tabulations are not available here.
  - Requested $limit=50,000 explicitly. Socrata's default is 1,000 rows, which is how a client silently returns a truncated table.
  - Digest pinned by the notebook and verified on this run (canonical-json). The publisher serves no prior vintage; this pin does not restore one, it guarantees a change cannot pass unnoticed.


Thirteen columns, 6,390 rows, one indicator — and a year axis that is **not** what a
span implies. The table runs 1950 to 2018, a span of sixty-eight years, but it carries only
**42 year-points**: 1950, 1960, 1970, 1980, and then every year from 1981.

Three of those intervals are ten years wide and thirty-eight are one year wide. Plotted against
an index rather than against `year`, the first four points occupy a tenth of the axis and stand
for four decades. That is a defect that survives every other filter in this notebook, because it
is in the axis rather than in the rows.

There is also a second thing in the header worth flagging now: `unit` is not metadata. It holds
**two different rate definitions**, and section 4 shows what that does.

## 3. What `stub_name` holds

In [3]:
"""What is actually in the stub column?"""
levels = describe_stub_levels(deaths)
print(levels.to_string(index=False, formatters={
    "rows": "{:,.0f}".format, "with_estimate": "{:,.0f}".format,
    "first_year": "{:,.0f}".format, "last_year": "{:,.0f}".format}))
print()
print(f"distinct classification schemes in one column: {len(levels)}")
print(f"rows with no estimate at all                 : "
      f"{int(deaths[ESTIMATE].isna().sum()):,} of {len(deaths):,}")


                                          stub_name  rows  single_race first_year last_year with_estimate
                                  Sex, age and race 1,596        False      1,950     2,018         1,282
              Sex, age and race and Hispanic origin 1,328        False      1,950     2,018           952
                                        Sex and age 1,176        False      1,950     2,018         1,174
                                       Sex and race   672        False      1,950     2,018           624
                   Sex and race and Hispanic origin   664        False      1,950     2,018           512
                                                Age   588        False      1,950     2,018           588
                                                Sex   168        False      1,950     2,018           168
                                              Total    84        False      1,950     2,018            84
Sex, age and race and Hispanic origin (Single 

**Twelve distinct classification schemes**, not twelve strata.

`Sex and age` is a cross-classification. So is `Sex, age and race and Hispanic origin`. They are
not levels of one hierarchy that could be summed or averaged together — they are separate
partitions of the same deaths, published side by side in one column, with `Total` as a
thirteenth thing that is neither.

Note also the year spans. The four `(Single race)` levels exist only in **2018**; everything else
runs from 1950. A groupby on `stub_name` without reading that column produces series of wildly
different lengths and no warning.

**[rule]** Before grouping on a classification column, list its distinct values and their spans.
If the values name *different schemes* rather than *levels of one scheme*, no aggregate over the
table is meaningful without a filter, and the filter is not optional.

## 4. How much of the table is the population?

In [4]:
"""How much of the table is the population, and which rate is "the" rate?"""
is_total = deaths["stub_name"] == NCHS_TOTAL_STUB
print(f"rows that are the whole population : {int(is_total.sum()):,} "
      f"({is_total.mean():.1%})")
print(f"rows that are a subgroup           : {int((~is_total).sum()):,} "
      f"({(~is_total).mean():.1%})")
print()

#: `unit` crosses `stub_name`: every scheme is published under two rate definitions.
print("rows by rate definition:")
print(deaths["unit"].value_counts().to_string())
print()

year = int(deaths.loc[is_total, "year"].max())
latest = deaths.loc[is_total & (deaths["year"] == year)]
print(f"the {year} rows for the whole population:")
for _, row in latest.iterrows():
    print(f"  {row[ESTIMATE]:>5.2f}   {row['unit']}")
print()

adjusted = latest.loc[latest["unit"].str.endswith("age-adjusted"), ESTIMATE].iloc[0]
crude = latest.loc[latest["unit"].str.endswith("crude"), ESTIMATE].iloc[0]
naive = deaths[ESTIMATE].mean()
print(f"mean of every estimate in the table : {naive:.2f}")
print(f"  vs age-adjusted {year}             : {adjusted:.2f}  "
      f"({(adjusted - naive) / adjusted:.1%} low)")
print(f"  vs crude {year}                    : {crude:.2f}  "
      f"({(crude - naive) / crude:.1%} low)")
print()
print("The first figure is not a rate for anything. It averages the whole")
print("population together with sex-by-age-by-race-by-Hispanic-origin cells,")
print("and age-adjusted rates together with crude ones, weighting each equally.")


rows that are the whole population : 84 (1.3%)
rows that are a subgroup           : 6,306 (98.7%)

rows by rate definition:
unit
Deaths per 100,000 resident population, crude           5578
Deaths per 100,000 resident population, age-adjusted     812

the 2018 rows for the whole population:
  14.20   Deaths per 100,000 resident population, age-adjusted
  14.80   Deaths per 100,000 resident population, crude

mean of every estimate in the table : 13.71
  vs age-adjusted 2018             : 14.20  (3.5% low)
  vs crude 2018                    : 14.80  (7.4% low)

The first figure is not a rate for anything. It averages the whole
population together with sex-by-age-by-race-by-Hispanic-origin cells,
and age-adjusted rates together with crude ones, weighting each equally.


**84 rows — 1.3%.** The other 98.7% are subgroup estimates.

So the default behaviour of almost any tool, given this table, is wrong. `df.estimate.mean()`
returns **13.71**, against an age-adjusted 2018 rate of **14.20** — **3.5% low**.

And note what the `unit` breakdown shows: there is no single "the 2018 rate". There are two,
**14.20 age-adjusted and 14.80 crude**, and they are stacked in the same column under the same
`stub_name`. A filter to `Total` alone still returns two rows per year, every year. Selecting one
of them by position — `.iloc[0]` — works here and is luck, not method.

The gap between 13.71 and 14.20 is small enough to be the most dangerous part. It is not
obviously wrong. A number that came out at 300 would be caught; one that comes out three and a
half percent low looks like a result.

**[rule]** A classification column is rarely the only one. Check whether a second column crosses
it — here `unit` crosses all twelve schemes — before treating a filter on the first as sufficient.

## 5. One year, counted repeatedly

In [5]:
"""One year, counted repeatedly."""
year = int(deaths["year"].max())
window = deaths[deaths["year"] == year]
by_level = window.groupby("stub_name").size().sort_values(ascending=False)
print(f"rows describing {year} alone: {len(window)}")
print(by_level.to_string())
print()
print(f"Every one of those {len(window)} rows describes the same twelve months.")
print(f"Only {int((window['stub_name'] == NCHS_TOTAL_STUB).sum())} of them are the "
      f"population; the rest re-partition it.")


rows describing 2018 alone: 276
stub_name
Sex, age and race and Hispanic origin (Single race)    48
Sex, age and race and Hispanic origin                  40
Sex, age and race                                      38
Sex, age and race (Single race)                        30
Sex and age                                            28
Sex and race and Hispanic origin (Single race)         24
Sex and race and Hispanic origin                       20
Sex and race                                           16
Age                                                    14
Sex and race (Single race)                             12
Sex                                                     4
Total                                                   2

Every one of those 276 rows describes the same twelve months.
Only 2 of them are the population; the rest re-partition it.


**276 rows describe 2018.** Two of them are the population — a crude rate and an
age-adjusted one. The remaining 274 re-partition the same deaths eleven different ways.

The largest single scheme in that year is `Sex, age and race and Hispanic origin (Single race)`
at 48 rows, which is *more* rows than the legacy `Sex, age and race and Hispanic origin` at 40 —
so even a reader who noticed the duplication and kept "the biggest breakdown" would pick the
standard that exists for one year only.

## 6. Two race-coding standards

In [6]:
"""Two race-coding standards, one changeover year."""
single = deaths["stub_name"].str.contains(NCHS_SINGLE_RACE_MARKER, regex=False)
overlap = (deaths.assign(single_race=single)
           .groupby(["year", "single_race"]).size().unstack(fill_value=0))
both = overlap[(overlap.get(True, 0) > 0) & (overlap.get(False, 0) > 0)]

print(f"rows on the legacy standard        : {int((~single).sum()):,}")
print(f"rows on the single-race standard   : {int(single.sum()):,}")
print(f"years using the single-race standard: "
      f"{sorted(int(y) for y in deaths.loc[single, 'year'].unique())}")
print()
print("years in which BOTH standards are published:")
print(both.to_string())


rows on the legacy standard        : 6,276
rows on the single-race standard   : 114
years using the single-race standard: [2018]

years in which BOTH standards are published:
single_race  False  True 
year                     
2018           162    114


**114 rows on the single-race standard, all in 2018**, against 6,276 on the legacy
one — and **2018 is the year both are published**, 162 legacy rows beside 114 new ones.

This is the same structure handbook §1.7 documents for HUD CHAS, where a category was retired and
the column name did not change. Here the marker is in the value rather than absent, which is
better — but only if you read it. A series that runs `Sex and race` through 2017 and then takes
whichever 2018 rows are available has changed its definition mid-series, and the discontinuity
will look like an event.

**[rule]** When a classification standard changes, the overlap year is the only place the two can
be compared and it is not a place they can be combined. Pick one standard, state it, and end the
series where it ends.

## Limitations

1. **This notebook is about a table's structure.** It makes no claim about suicide rates, their
   trend, their causes or any policy bearing on them, and nothing here should be read that way.

2. **The series ends in 2018.** This resource is not maintained to the present, so nothing here
   is current, and the corpus should not cite it as a recent figure.

3. **This is not a WONDER substitute.** Custom cross-tabulations, sub-national geography and
   suppression-controlled small cells are WONDER features that this API does not provide.

4. **Age-adjusted and crude rates are stacked in one column.** Section 4 separates them and
   reports both. This notebook does not say which is appropriate — that depends on the question,
   and the two answer different ones.

5. **`describe_stub_levels` reports structure, not validity.** It says which schemes are present.
   It cannot say which is appropriate for a given question.

6. **No margins.** NCHS publishes standard errors for many of these rates elsewhere; this
   resource does not carry them, so nothing here supports a significance claim.

7. **The single-race overlap is one year in this table.** Other NCHS tables have longer overlaps,
   and the handling in section 6 is a rule rather than a measurement that transfers.

8. **906 of 6,390 estimate cells are null**, concentrated in the finest breakdowns. This notebook
   counts rows, not populated cells; an analysis over any deep scheme would have to account for
   them.

**On vintage.** The Socrata resource serves no version and can be revised in place, so this retrieval is the only vintage that will exist; the payload hash is the only identifier. This notebook therefore records `vintage_pinned: false` (handbook §5.2); the payload hash and retrieval time in its provenance record are what a later reader has to compare against.

## Key Takeaways

1. **WONDER refuses automated query; the Socrata API does not** — and it is not a substitute for
   it.

2. **`stub_name` holds twelve classification schemes, not twelve strata.** They are separate
   partitions of the same deaths.

3. **Only 84 of 6,390 rows (1.3%) are the whole population.**

4. **`unit` crosses all twelve schemes.** Filtering to `Total` still returns two rows per year —
   14.20 age-adjusted and 14.80 crude for 2018. Picking one by position is luck.

5. **A naive mean gives 13.71 against 14.20 age-adjusted** — three and a half percent low, which
   is more dangerous than obviously wrong.

6. **276 rows describe 2018**; two of them are the population.

7. **Two race-coding standards overlap in 2018** — 162 legacy rows beside 114 single-race. An
   overlap year is where they can be compared, not combined.

8. **Sixty-eight years of span, 42 year-points.** Decadal to 1980, annual after. Plot against
   `year`, never against an index.

9. **Pass `$limit` explicitly and refuse an exactly-full page.** Socrata's 1,000-row default is
   how a table silently truncates.

## Exercises

1. **Build the series properly.** Extract the age-adjusted total rate from 1950 to 2018 as a
   single clean series, stating every filter you applied — including the one on `unit` — and why
   each is necessary.

2. **Quantify the naive-mean error across years.** Compute the unfiltered mean per year against
   the total for that year, and report whether the gap is stable or grows as more breakdowns are
   added.

3. **Draw the axis wrong, then right.** Plot the total series against its positional index and
   against `year`. Describe what the first plot implies about 1950–1980 that the data does not
   support.

4. **Find the other tables like this.** Search `data.cdc.gov` for other *Health, United States*
   resources with a `stub_name` column and report how many share this structure.

5. **Handle the standard change.** Construct the longest defensible race-specific series you can
   from this table, and say exactly where it must stop and why.

6. **Test the truncation guard.** Call the fetcher with `limit=100` and read the error. Then say
   what a client that did not raise would have returned, and how you would have noticed.

7. **Write the column note.** In two sentences each, draft what `stub_name` and `unit` should say
   in their documentation to stop a reader averaging over them.

## References

- National Center for Health Statistics. *Health, United States* — death rates for suicide.
  https://data.cdc.gov/d/9j2v-jamp
- Centers for Disease Control and Prevention. *CDC WONDER.* https://wonder.cdc.gov/
- Office of Management and Budget (1997). *Revisions to the Standards for the Classification of
  Federal Data on Race and Ethnicity* — the change behind the single-race coding.
  https://obamawhitehouse.archives.gov/omb/fedreg_1997standards
- Socrata Open Data API. *Paging through data.* https://dev.socrata.com/docs/paging.html
- Handbook §1.7 (HUD CHAS — retired categories under a stable column name), §3 (denominators,
  universes and partitions).
- D20 (`05-social-systems/D20_device_access`) — two partitions in one flat list.
- `src/krl_tutorials_data/sources.py` (`fetch_cdc_socrata`, `describe_stub_levels`) and
  `tests/test_cdc_socrata.py`.

**If this subject matter is affecting you personally**, support is available — in the United
States, the 988 Suicide and Crisis Lifeline can be reached by call or text at 988.